# 01 — Exploración de Datos

Exploración inicial del dataset WHO GSHS El Salvador 2013.

**Objetivos:**
- Verificar forma del dataset y tipos de datos
- Identificar y cuantificar valores sentinel (`1.79769313486232e+308`)
- Analizar missingness por columna
- Visualizar distribución de variables objetivo (targets)
- Examinar balance de clases para clasificación

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_raw, get_data_summary
from src.config import SENTINEL_VALUE, CLASSIFICATION_ALL_TARGETS

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Cargar datos

In [ ]:
df_raw = pd.read_csv('../data/SLV2013_Public_Use.csv')
print(f'Shape sin limpieza: {df_raw.shape}')
print(f'Celdas con sentinel: {(df_raw == SENTINEL_VALUE).sum().sum()}')

In [ ]:
df = load_raw('../data/SLV2013_Public_Use.csv')
print(f'Shape después de reemplazar sentinel: {df.shape}')
print(f'Valores nulos totales: {df.isnull().sum().sum()}')

## 2. Resumen por columna

In [ ]:
summary = get_data_summary(df)
print('Columnas con mayor missingness:')
summary.head(20)

## 3. Heatmap de valores faltantes

In [ ]:
from src.visualization.plots import plot_missing_heatmap
fig = plot_missing_heatmap(df)
plt.show()

## 4. Distribución de variables objetivo

In [ ]:
from src.data.clean import encode_binary_targets
from src.visualization.plots import plot_target_distribution

df_recoded = encode_binary_targets(df, CLASSIFICATION_ALL_TARGETS)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, target in zip(axes.flatten(), CLASSIFICATION_ALL_TARGETS):
    if target in df_recoded.columns:
        counts = df_recoded[target].value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values)
        ax.set_title(target)
        ax.set_xlabel('Clase')
        ax.set_ylabel('Conteo')
plt.suptitle('Distribución de Variables Objetivo (0=No, 1=Sí)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Variables continuas (Q4 altura, Q5 peso)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Q4'].dropna().hist(ax=axes[0], bins=30, edgecolor='black')
axes[0].set_title('Q4 — Altura (m)')
df['Q5'].dropna().hist(ax=axes[1], bins=30, edgecolor='black', color='salmon')
axes[1].set_title('Q5 — Peso (kg)')
plt.show()